In [4]:
pip install espnet soundfile espnet_model_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 9.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Local Inference on GPU
Model page: https://huggingface.co/espnet/owsm_v4_small_370M

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/espnet/owsm_v4_small_370M)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [1]:
import soundfile as sf
import os

# Create a dummy WAV file for demonstration if it doesn't exist
if not os.path.exists('speech.wav'):
    # Generate a simple sine wave as an example
    samplerate = 16000  # Sample rate (Hz)
    duration = 3      # Duration (seconds)
    frequency = 440   # Frequency (Hz)

    import numpy as np
    t = np.linspace(0., duration, int(samplerate * duration), endpoint=False)
    amplitude = 0.5
    data = amplitude * np.sin(2. * np.pi * frequency * t)

    # Save to 'speech.wav'
    sf.write('speech.wav', data, samplerate)
    print("Generated a sample 'speech.wav' file.")
else:
    print("Using existing 'speech.wav' file.")

Using existing 'speech.wav' file.


In [2]:
import urllib.request

# URL to a sample wav file containing actual speech
url = "https://www.kozco.com/tech/LRMonoPhase4.wav"
speech_file = "speech.wav"

print(f"Downloading sample speech from {url}...")
urllib.request.urlretrieve(url, speech_file)
print(f"Saved as {speech_file}. You can now run the transcription cell again.")

Saved as speech.wav. You can now run the transcription cell again.


In [3]:
from espnet_model_zoo.downloader import ModelDownloader
from espnet2.bin.asr_inference import Speech2Text
import yaml
import tempfile
import os
import glob

# 1. Download model files
d = ModelDownloader()
kwargs = d.download_and_unpack("espnet/owsm_v4_small_370M")

# 2. Identify the config file path
# Often download_and_unpack returns the path in 'asr_train_config' or 's2t_train_config'
config_path = kwargs.get('asr_train_config') or kwargs.get('s2t_train_config')

if not config_path:
    # Fallback: search for config.yaml in the downloaded directories
    search_path = os.path.join(d.cachedir, "**", "config.yaml")
    found_configs = glob.glob(search_path, recursive=True)
    if found_configs:
        config_path = found_configs[0]

# 3. Clean the config file
if config_path:
    with open(config_path, 'r') as f:
        config_data = yaml.safe_load(f)

    def remove_key_recursive(obj, key):
        if isinstance(obj, dict):
            obj.pop(key, None)
            for v in obj.values():
                remove_key_recursive(v, key)
        elif isinstance(obj, list):
            for item in obj:
                remove_key_recursive(item, key)

    print(f"Cleaning config at {config_path}...")
    remove_key_recursive(config_data, 'sym_na')

    # Create a new temporary config file
    fd, temp_path = tempfile.mkstemp(suffix=".yaml")
    with os.fdopen(fd, 'w') as f:
        yaml.safe_dump(config_data, f)

    # Update kwargs for initialization
    kwargs['asr_train_config'] = temp_path

# 4. Final adjustments to kwargs keys
if 's2t_model_file' in kwargs:
    kwargs['asr_model_file'] = kwargs.pop('s2t_model_file')
if 's2t_train_config' in kwargs:
    del kwargs['s2t_train_config']

# Filter out any keys that Speech2Text doesn't accept
# (Sometimes model_zoo adds extra metadata keys)

print("Initializing model...")
try:
    model = Speech2Text(**kwargs)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Failed: {e}")
    raise e

Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.
/usr/local/lib/python3.12/dist-packages/espnet2/enh/encoder/stft_encoder.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/usr/local/lib/python3.12/dist-packages/espnet2/enh/layers/uses2_swin.py:329: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

Cleaning config at /usr/local/lib/python3.12/dist-packages/espnet_model_zoo/models--espnet--owsm_v4_small_370M/snapshots/1d855cbc60d16362a281444be4849539e17af077/exp/s2t_train_conv2d8_size768_e9_d9_mel128_raw_bpe50000/config.yaml...
Initializing model...
Model loaded successfully!


In [ ]:
import soundfile as sf

# Load the new audio file containing actual speech
speech, rate = sf.read('speech.wav')

# Perform inference
print("Transcribing real speech sample...")
results = model(speech)

if results:
    text, *_ = results[0]
    print(f"\nTranscribed Text: {text}")
else:
    print("No transcription results found.")

Transcribing real speech sample...


/usr/local/lib/python3.12/dist-packages/espnet2/asr/espnet_model.py:402: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(self.autocast_frontend, dtype=autocast_type):
